# Generate vectors for cells for an arbitrary scale

The inisial dataset you need for this:
 1) adjacency matrix
 2) coordinated of the nodes
 3) the shapefile of the boundry of your map - areas 
The paramether needed
 The parameter named "lam" used in the initialize function specifies the desired spatial scale—that is, the number of cells along each dimension of the grid.
 
coordinates is a n by 2 array where the i'th entry is the coordinates of the i'th city [x, y]
adjacency is a n by n array where the ij'th entry is the weight of the vector from city i to city j
lam stands for lambda which is the number of cells in a row
method should be 'sum' or 'mean'
both of coordinates and adjacency are given as numpy arrays

In [1]:
def create_vectors_interpolate(adjacency, coordinates, method='sum'):
    # n is the number of nodes
    n = len(adjacency)

    # vecs is the array such that its ij'th entry ([i, j]) is the vector ascociated with the center of the cell at row i and coloumn j
    vecs = np.zeros((w, h, 2))

    # num[i, j] = number of active outgoing vectors from center [i, j]
    weight = np.zeros((w, h))
    exist_node = np.zeros((w, h))

    for i in range(n):
        # the i'th node is in the cell index by [p, q] where p and q are as follows:
        p = min(math.floor((coordinates[i][0] - min_map_x) / l) + 1, w - 2)
        q = min(math.floor((coordinates[i][1] - min_map_y) / l) + 1, h - 2)
        exist_node[p, q] = 1
        for j in range(n):
        # vector from city i to city j
            new_vec = coordinates[j] - coordinates[i]
            # adding new_vec with its weight starting from the correct cell
            vecs[p, q] += adjacency[i, j] * new_vec

        # there is a new nonzero vector outgoing from cell [p, q]
            # if adjacency[i, j] != 0:
            weight[p, q] += adjacency[i, j]

    # if the method is mean, we devide the vector ascociated with center [p, q] by the number of active vectors from that cell
    if method == 'mean':
        for p in range(w):
            for q in range(h):
                if weight[p,q] != 0:
                    vecs[p, q] = vecs[p, q] / weight[p, q]

    new_coords = []
    new_vecs = []
    for p in range(w):
        for q in range(h):
            if exist_node[p, q] == 1:
                new_coords.append((p, q))
                new_vecs.append((vecs[p, q, 0], vecs[p, q, 1]))


    print(len(new_coords))
    return new_coords, new_vecs



def initialize(coordinates, lam=4):# lam is the scale you are interested in to have (number od cell in each dimention)
    global h, w, min_map_x, max_map_x, min_map_y, max_map_y, l

    min_map_x = np.min(coordinates[:, 0])
    max_map_x = np.max(coordinates[:, 0])
    min_map_y = np.min(coordinates[:, 1])
    max_map_y = np.max(coordinates[:, 1])

    lam = lam - 2
    l = (max_map_x - min_map_x) / lam
    w = lam + 2
    h = math.ceil((max_map_y - min_map_y) / l) + 2

    global xy

    idw = np.linspace(0, w - 1, w)
    idh = np.linspace(0, h - 1, h)
    y, x = np.meshgrid(idh, idw)
    x, y = x.astype(int), y.astype(int)
    xy = np.dstack([x, y])


    print(xy.shape)
    
    
def find_n_points_on_boundry(gdf, n=100):
    #Load the shapefile
    #gdf = gpd.read_file('th878nx5786.shp')

    #assuming there's only one geometry in the GeoDataFrame
    geometry = gdf.geometry.iloc[0]

    boundary = geometry.boundary
    boundary_length = boundary.length

    distances = np.linspace(0, boundary_length, n)

    #generate n points along the boundary
    points = [boundary.interpolate(distance) for distance in distances]
    points_list = [(point.x, point.y) for point in points]
    
    return points_list
    
    
def add_boundry_points_to_coords_vecs(coords, vecs, gdf):
    
    points_list= find_n_points_on_boundry(gdf, n=100)
    
    
    #make an array from coords and vecs
    coords = np.array(coords)
    vecs = np.array(vecs)
    
    for p in points_list:
        cell_x = min(math.floor((p[0] - min_map_x) / l) + 1, w - 2)
        cell_y = min(math.floor((p[1] - min_map_y) / l) + 1, h - 2)
        coords = np.append(coords, [[cell_x, cell_y]], axis=0)
        vecs = np.append(vecs, [[0, 0]], axis=0)
        
        
    return coords, vecs
        
 ##################################################################new code   
# make triangles   
def interpolate(center, corners, vectors, eps_area=1e-12):
    """
    Interpolates a vector at `center` using corners of a triangle and their vectors,
    based on areas of sub-triangles for numerical stability.
    
    Parameters:
        center: (2,) array-like, point to interpolate
        corners: (3,2) array, triangle vertex coordinates
        vectors: (3,2) array, vectors at triangle vertices
        eps_area: minimum triangle area to avoid unstable weights
    Returns:
        interpolated vector: (2,)
    """
    center = np.asarray(center, dtype=float)
    corners = np.asarray(corners, dtype=float)
    vectors = np.asarray(vectors, dtype=float)
    
    # Compute sub-triangle areas
    def tri_area(a, b, c):
        return 0.5 * abs(np.cross(b - a, c - a))
    
    A0 = tri_area(center, corners[1], corners[2])
    A1 = tri_area(center, corners[0], corners[2])
    A2 = tri_area(center, corners[0], corners[1])
    
    total_area = A0 + A1 + A2
    if total_area <= eps_area:
        # Degenerate triangle: fallback to mean vector
        return np.mean(vectors, axis=0)
    
    # weights
    w = np.array([A0, A1, A2]) / total_area
    
    # interpolated vector
    vp = w[0]*vectors[0] + w[1]*vectors[1] + w[2]*vectors[2]
    return vp

def find_field(coords, vecs, debug=False):
    """
    Compute vector field for all grid cells based on Delaunay triangulation.
    Returns a (w,h,2) array of interpolated vectors.
    """
    coords = np.asarray(coords)
    vecs = np.asarray(vecs)
    
    tri = Delaunay(coords)
    field = np.zeros((w, h, 2))
    
    for i in range(w):
        for j in range(h):
            p = xy[i,j]  # grid point coordinate
            simplex_idx = tri.find_simplex(p)
            
            if simplex_idx == -1:
                # point outside triangulation: use nearest neighbour
                dists = np.linalg.norm(coords - p, axis=1)
                nn = np.argmin(dists)
                field[i,j] = vecs[nn]
            else:
                corner_indices = tri.simplices[simplex_idx]
                corners = coords[corner_indices]
                vectors_tri = vecs[corner_indices]
                field[i,j] = interpolate(p, corners, vectors_tri)
    
    return field
###################################################################################################


def find_vector_field(coords, vecs):
    # coords, vecs = add_corners(coords, vecs)
    field = find_field(coords, vecs)
    return field

def find_field_point_inside_boundry(gdf, field):
    polygon_boundry= gdf['geometry'][0]

    real_field = field.copy()

    for i in range(len(field)):
        for j in range(len(field[0])):
            real_x = i * l + min_map_x # --> -46.2394723847 
            real_y = j * l + min_map_y
            # print(real_x, real_y)?
            point = Point(real_x, real_y)
            if polygon_boundry.contains(point) is True:
                u, v = real_field[i, j]
                real_field[i, j] = [u * l, v * l]
            else:
                real_field[i, j] = [0, 0]
            
    return real_field

    
    
def find_real_coordinates_and_field(coords, gdf, field):
    real_coordinates = coords.copy().astype(float)
    for i in range(len(real_coordinates)):
        x, y = real_coordinates[i]
        real_coordinates[i] = [x * l + min_map_x, y * l + min_map_y]
    
    real_field=find_field_point_inside_boundry(gdf, field) 
  
    
    return real_coordinates, real_field



#plot delanay for sanity check
def plot_delanay(coords):
    
    tri = Delaunay(coords)
    plt.triplot(coords[:, 0], coords[:, 1], tri.simplices)
    plt.plot(coords[:, 0], coords[:, 1], 'o')
    plt.show()



 

 

# Find critical points - sinks sources - contours

### To identify critical points (such as sinks, sources, and saddle points), two parameters must be set in the functions find_sinks, find_sources, and find_contours:
##### tr: Threshold for vector magnitude. If the magnitude at a point is below this threshold, the point is considered a potential critical point.
##### tr_jacob: Threshold for the Jacobian value, used to classify the type of critical point (e.g., sink, source, or saddle point).



In [2]:

def is_sink(jacob,threshold_jacob):
    trace = jacob[0, 0] + jacob[1, 1]
    det = jacob[0, 0] * jacob[1, 1] - jacob[0, 1] * jacob[1, 0]
    return trace < -threshold_jacob and det > threshold_jacob

# trace > threshold_jacob  it should be trace <0, trace < -threshold_jacop
# lambda 1 and lambda 2 negative
# lambda 1 * lambda 2 >0 > threshold^2
#delta>0 delta positive
#delta= trace^2 - 4det

#atracting focus:
#trace > threshold_jacob  it should be trace <0, trace < -threshold_jacop . imaginary parts delete each other out
#det R1*R2 + I1^2 - det >0>> threshold^2
#delta <0 negative

#source
#trace > threshold_jacob

#onother way find the exact lamda

def jacobian(field, point):
    jacob = np.array([[0.0, 0.0], [0.0, 0.0]])
    x, y = point
    jacob[0, 0] = field[x + 1, y, 0] - field[x - 1, y, 0]
    jacob[1, 0] = field[x + 1, y, 1] - field[x - 1, y, 1]
    jacob[0, 1] = field[x, y + 1, 0] - field[x, y - 1, 0]
    jacob[1, 1] = field[x, y + 1, 1] - field[x, y - 1, 1]
    return jacob


def find_critical_points(field, threshold):
    magnitude = np.sqrt(field[:, :, 0]**2 + field[:, :, 1]**2)
    indices = np.where(magnitude[1:-1, 1:-1] < threshold)
    return np.dstack([indices[0], indices[1]])[0] + [1, 1]


def find_sinks(field, tr, tr_jacob):
    critical_points = find_critical_points(field, tr)
    sinks = []
    for point in critical_points:
        jacob = jacobian(field, point)
        if is_sink(jacob,tr_jacob):
            sinks.append(point)
    return sinks

def is_source(jacob,threshold_jacob):
    trace = jacob[0, 0] + jacob[1, 1]
    det = jacob[0, 0] * jacob[1, 1] - jacob[0, 1] * jacob[1, 0]
    return trace > threshold_jacob and det > threshold_jacob

def find_sources(field, tr, tr_jacob):
    critical_points = find_critical_points(field, tr)
    sources = []
    for point in critical_points:
        jacob = jacobian(field, point)
        if is_source(jacob,tr_jacob):
            sources.append(point)
    return sources



#find three structures: Repelling focus, Attracting focus, and center
def is_contour(jacob,threshold_jacob):
    trace = jacob[0, 0] + jacob[1, 1]
    det = jacob[0, 0] * jacob[1, 1] - jacob[0, 1] * jacob[1, 0]
    return trace*trace - 4*det < -threshold_jacob



def find_contours(field, tr, tr_jacob):
    critical_points = find_critical_points(field, tr)
    contours = []
    for point in critical_points:
        jacob = jacobian(field, point)
        if is_contour(jacob,tr_jacob):
            contours.append(point)
    return contours

# Function to approximate the point p based on cell_x, cell_y, with bounds
def find_coords_critical_ponits(cell_x, cell_y, l, min_map_x, min_map_y, w, h):
    import math
    
    p_x = max(min_map_x, min((cell_x - 1) * l + min_map_x + l / 2, min_map_x + (w - 1) * l))
    p_y = max(min_map_y, min((cell_y - 1) * l + min_map_y + l / 2, min_map_y + (h - 1) * l))
    
    return (p_x, p_y)

def find_coords_critical_point_another_way(p, q, l, min_map_x, min_map_y, w, h):
    x = (p - 1) * l + min_map_x
    y = (q - 1) * l + min_map_y
    if x > (w - 2) * l + min_map_x:
        x = (w - 2) * l + min_map_x
    
    
    if y > (h - 2) * l + min_map_y:
        y = (h - 2) * l + min_map_y
        
    return (x, y)


def find_coords_critical_points_real(points_coords, gdf):
    polygon_boundry= gdf['geometry'][0]

    real_critical_points=[]
    for p in points_coords:
        real_x= p[0]
        real_y= p[1]
        point = Point(real_x, real_y)
        if polygon_boundry.contains(point) is True:
            real_critical_points.append((real_x, real_y))
            
    return real_critical_points


def find_coordinates_critical_points(critical_points, gdf_b):
    points_locs=[]
    for p in critical_points:
    #print(p)
        p_xy= find_coords_critical_ponits(p[0], p[1], l, min_map_x, min_map_y, w, h)
        #p_xy= find_coords_critical_point_another_way(p[0], p[1], l, min_map_x, min_map_y, w, h)
    #print(p_xy)
        points_locs.append(p_xy)
    
    #find points inside the boundry
    points_locs= find_coords_critical_points_real(points_locs, gdf_b)
        
    return points_locs
    

# remove boundary effect and visualise the sinks and sources

In [6]:
# remove the effect of boundary - remove points that have quiet small size vector on boundary and near boundary
def shrink_polygon(polygon, scale_factor):
    """
    Shrinks a polygon towards its centroid while maintaining shape.
    
    Parameters:
        polygon (shapely.geometry.Polygon): The original polygon.
        scale_factor (float): The scaling factor (e.g., 0.8 for 80% of original size).
    
    Returns:
        shapely.geometry.Polygon: The shrunken polygon.
    """
    # Find centroid
    centroid = polygon.centroid
    cx, cy = centroid.x, centroid.y

    # Scale each point towards the centroid
    new_coords = [
        ((x - cx) * scale_factor + cx, (y - cy) * scale_factor + cy)
        for x, y in polygon.exterior.coords
    ]

    return Polygon(new_coords)


def get_inside_points_list(points_list, polygon):
    """
    Filters points that lie inside the given polygon for each list of points.
    """
    inside_points = [Point(x, y) for x, y in points_list if polygon.contains(Point(x, y))]
    inside_points_array = np.array([(p.x, p.y) for p in inside_points])

    return inside_points_array

def compute_density_map(inside_point, gdf, grid_size=200):
    """
    Computes a KDE-based density map for a list of inside points.

    """
    x_min, y_min, x_max, y_max = gdf.bounds
    x_grid = np.linspace(x_min, x_max, grid_size)
    y_grid = np.linspace(y_min, y_max, grid_size)
    X, Y = np.meshgrid(x_grid, y_grid)
    grid_points = np.vstack([X.ravel(), Y.ravel()]).T

    if inside_points.shape[0] < 2:
        # Not enough points to compute KDE
        Z = np.full(X.shape, np.nan)
        return Z

    kde = gaussian_kde(inside_points.T)
    Z = kde(grid_points.T).reshape(X.shape)

    # Mask values outside the polygon
    mask = np.array([gdf.contains(Point(x, y)) for x, y in grid_points]).reshape(X.shape)
    Z[~mask] = np.nan

    return Z